In [1]:
from pathlib import Path
import os

import datasets
import pandas as pd
import numpy as np
from peft import LoraConfig
from transformers import AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig, DataCollatorWithPadding
from transformers.integrations import WandbCallback
from trl import SFTTrainer, SFTConfig
import torch
from tqdm import tqdm
import wandb

import sys
sys.path.append('../../../../')
from task.prop import PropTask, full_text_ds_path, or_text_ds_path

# model_name = "Qwen/Qwen2.5-Coder-7B"
model_name = "Qwen/Qwen2.5-Coder-0.5B"

with (Path(os.environ['HOME']) / 'wandb.txt').open() as fp:
    key = fp.read().strip()

os.environ['WANDB_API_KEY'] = key
os.environ['WANDB_PROJECT'] = 'qwen'
os.environ['WANDB_LOG_MODEL'] = 'false'

wandb.login()


def score(preds, labels, succ_id, fail_id):
    is_true = torch.argmax((labels == succ_id).int(), axis=-1) > 0

    t = torch.argmax((preds == succ_id).int(), axis=-1)
    f = torch.argmax((preds == fail_id).int(), axis=-1)

    pred_is_true = (t != 0) * ((f == 0) + (t < f))
    pred_is_false = (f != 0) * ((t == 0) + (f < t))

    true_pos = is_true * pred_is_true
    true_neg = (~is_true) * pred_is_false
    false_pos = (~is_true) * pred_is_true
    false_neg = is_true * pred_is_false

    true_pos = torch.mean(true_pos.float())
    true_neg = torch.mean(true_neg.float())
    false_pos = torch.mean(false_pos.float())
    false_neg = torch.mean(false_neg.float())
    
    return {
        'gen_acc': true_pos + true_neg,
        # 'true_pos': true_pos,
        # 'true_neg': true_neg,
        # 'false_pos': false_pos,
        # 'false_neg': false_neg
    }

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: wtong98 (wtong98-harvard-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [2]:
def make_ds(depth, split):
    task = PropTask(depth=depth, split=split, cot='text', ds_path=or_text_ds_path)
    task.load_ds()

    ds = datasets.concatenate_datasets([task.true_ds, task.false_ds]).shuffle()

    # temporarily reduce size for debugging
    ds = ds.select(range(50000))

    # TODO: reformat permanently in dataset
    ds = ds.map(lambda x: {
        'prompt': x['prompt'] + x['completion'][:-11] + '|', # all except final classification
        'completion': '<success />' if x['is_true'] else '<failure />'
    }, num_proc=16)
    # ds = ds.rename_column('prompt', 'proposition')
    # ds = ds.remove_columns(['completion'])
    return ds


train_split = 4
test_splits = [4, 6, 8]
range_hops = [1] + [h + 1 for h in test_splits] + [np.inf]
ranges = list(zip(range_hops[:-1], range_hops[1:]))

train_ds = make_ds(train_split, 'train')
test_ds = make_ds(train_split, 'test')
test_ds = test_ds.select(range(100))  # TODO: should preferably incorporate logic by subsampling during training

val_ds_set = [make_ds(r, split='range') for r in ranges]

Map (num_proc=16):   0%|          | 0/50000 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/50000 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/50000 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/50000 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/50000 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/50000 [00:00<?, ? examples/s]

In [3]:
train_ds[-1]

{'is_true': True,
 'completion': '<success />',
 'length': 4,
 'ops': ['intro h', 'apply Or', 'exact'],
 'prompt': '<state id="0"><if>p1 p2 p3 : Prop</if><then>⊢ p1 → (p1 ∨ p2) ∨ p1 ∨ p1 ∨ p3 ∨ p2 ∨ p1 ∨ p1</then></state>|<tactic>intro h1</tactic><state id="1"><if>p1 p2 p3 : Prop</if><if>h1 : p1</if><then>⊢ (p1 ∨ p2) ∨ p1 ∨ p1 ∨ p3 ∨ p2 ∨ p1 ∨ p1</then></state><tactic>apply Or.inl</tactic><state id="2"><if>case h</if><if>p1 p2 p3 : Prop</if><if>h1 : p1</if><then>⊢ p1 ∨ p2</then></state><tactic>apply Or.inl</tactic><state id="3"><if>case h.h</if><if>p1 p2 p3 : Prop</if><if>h1 : p1</if><then>⊢ p1</then></state><tactic>exact h1</tactic><state id="4"><complete /></state>|'}

In [4]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4'
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map='auto',
    attn_implementation='flash_attention_2',
    quantization_config=quant_config
)

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",  # NOTE: potentially shady parameter
    target_modules=("q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")
)

args = SFTConfig(
    output_dir="~/scratch/qwen25_coder7b_prop_qlora", # TODO: pick destination
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,      
    learning_rate=2e-4,                  # QLoRA LR baseline
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=5,
    save_steps=1000,
    bf16=True,                           
    gradient_checkpointing=True,
    optim="adamw_bnb_8bit",
    # optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    weight_decay=0.0,
    completion_only_loss=True,
    packing=True,
    max_length=2048,
    eval_strategy='steps',
    torch_compile=False
)

trainer = SFTTrainer(
    model=model,
    peft_config=peft_config,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds
)

# TODO: generalize to evaluate on multiple dataset splits
class WandbEvalCallback(WandbCallback):
    def __init__(self, trainer, val_ds_set, num_samples=100):
        super().__init__()
        self.trainer = trainer
        self.tokenizer = self.trainer.processing_class
        self.val_ds_set = val_ds_set
        self.num_samples = num_samples
        
        self.succ_id = self.tokenizer.encode('success')[0]
        self.fail_id = self.tokenizer.encode('failure')[0]
        

    def on_evaluate(self, args, state, control, **kwargs):
        super().on_evaluate(args, state, control, **kwargs)
        
        all_res = {}
        for r, val_ds in tqdm(zip(ranges, self.val_ds_set), total=len(self.val_ds_set)):
            ds = val_ds.shuffle().select(range(self.num_samples))
            self.tokenizer.padding_side = 'left'
            coll = DataCollatorWithPadding(tokenizer=self.tokenizer)
            inp_ids = [self.tokenizer(text) for text in ds['prompt']]
            lab_ids = [self.tokenizer(text) for text in ds['completion']]
            inp = coll(inp_ids)
            lab = coll(lab_ids)
            self.tokenizer.padding_side = 'right'

            inp['input_ids'] = inp['input_ids'].to(device='cuda')
            inp['attention_mask'] = inp['attention_mask'].to(device='cuda')
            lab['input_ids'] = lab['input_ids'].to(device='cuda')

            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                # preds = trainer.model.generate(**inp, max_new_tokens=trainer.args.max_length)
                preds = trainer.model.generate(**inp, max_new_tokens=10)

            res = score(preds, lab['input_ids'], self.succ_id, self.fail_id)
            all_res[f'range_{r}'] = res
            
        self._wandb.log(all_res)

        
eval_callback = WandbEvalCallback(trainer, val_ds_set)
trainer.add_callback(eval_callback)

trainer.train()
wandb.finish()

Adding EOS to train dataset:   0%|          | 0/50000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/50000 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/50000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

...

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


/n/netscratch/pehlevan_lab/Lab/wlt/envs/venv_imply/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Casting fp32 inputs back to torch.bfloat16 for flash-attn compatibility.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
5,3.964500,1.701537,0.621627,318106.000000,0.524264
10,0.532400,0.254432,0.491510,635381.000000,0.981061
15,0.000100,0.409796,0.321799,952735.000000,0.981061
20,0.000000,0.433485,0.342554,1269989.000000,0.981061
25,0.000000,0.439063,0.420793,1587723.000000,0.981061
30,0.000000,0.437559,0.474835,1904425.000000,0.981061
35,0.000000,0.438547,0.508123,2219877.000000,0.981061
40,0.000000,0.441758,0.522544,2535339.000000,0.981061
45,0.000000,0.443873,0.528127,2852977.000000,0.981061
50,0.000000,0.448725,0.529665,3169780.000000,0.981061


 75%|███████▌  | 3/4 [00:02<00:00,  1.10it/s]


KeyboardInterrupt: 

In [5]:
wandb.finish()

eval/entropy,██▅▅▁▁▁▁▃▃▅▅▅▅▆▆▆▆▆▆▆▆▆▆
eval/loss,██▁▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂
eval/mean_token_accuracy,▁▁██████████████████████
eval/num_tokens,▁▁▂▂▂▂▃▃▄▄▄▄▅▅▅▅▆▆▇▇▇▇██
eval/runtime,██▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/samples_per_second,▁▁██████████████████████
eval/steps_per_second,▁▁██████████████████████
train/entropy,████▃▃▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂
train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇▇███
train/global_step,▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇▇██
+5,...


In [ ]:
trainer.processing_class('state.<|endoftext|>')

In [ ]:
trainer.processing_class

In [ ]:
ds = test_ds.select(range(10))
ids = [trainer.processing_class(text) for text in ds['text']]
tok_ds = datasets.Dataset.from_list(ids)

trainer.args.packing = True
pred = trainer.predict(tok_ds)

In [ ]:
from transformers import DataCollator

trainer.processing_class.padding_side = 'left'
inp = DataCollatorWithPadding(tokenizer=trainer.processing_class)(ids)
inp

In [ ]:
inp_ids_orig = inp['input_ids']
inp['input_ids'] = inp['input_ids'][:,:-500]
inp['input_ids']

In [ ]:
inp['attention_mask'].dtype

In [ ]:
with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
    inp['input_ids'] = inp['input_ids'].to(device='cuda')
    inp['attention_mask'] = inp['attention_mask'].to(device='cuda')
    out = trainer.model.generate(**inp, max_new_tokens=2048)

In [ ]:
trainer.processing_class.encode('<success />')

In [ ]:
succ_id = trainer.processing_class.encode('success')[0]
fail_id = trainer.processing_class.encode('failure')[0]

labels = DataCollatorWithPadding(tokenizer=trainer.processing_class)(ids)
labels = labels['input_ids'].to('cuda')

preds = out

def score(preds, labels, succ_id, fail_id):
    is_true = torch.argmax((labels == succ_id).int(), axis=-1) > 0

    t = torch.argmax((preds == succ_id).int(), axis=-1)
    f = torch.argmax((preds == fail_id).int(), axis=-1)

    pred_is_true = (t != 0) * ((f == 0) + (t < f))
    pred_is_false = (f != 0) * ((t == 0) + (f < t))

    true_pos = is_true * pred_is_true
    true_neg = (~is_true) * pred_is_false
    false_pos = (~is_true) * pred_is_true
    false_neg = is_true * pred_is_false

    true_pos = torch.mean(true_pos.float())
    true_neg = torch.mean(true_neg.float())
    false_pos = torch.mean(false_pos.float())
    false_neg = torch.mean(false_neg.float())
    
    return {
        'true_pos': true_pos,
        'true_neg': true_neg,
        'false_pos': false_pos,
        'false_neg': false_neg
    }


score(preds, labels, succ_id, fail_id)

In [ ]:
trainer.processing_class.batch_decode(out)[0]

In [ ]:
trainer.processing_class.decode(inp['input_ids'][0][:-700])